# FHIR Patient Data Quality Profiling

## Purpose

In this notebook, I profile the Silver FHIR Patient dataset and define the
data-quality rules that will later be enforced directly by the Lakeflow
Bronze-to-Silver transformation.

I am not creating another cleaned Patient table in this notebook.

The purpose of this stage is to:

- identify incomplete or invalid Patient records
- measure current quality-rule violations
- distinguish critical identifiers from optional FHIR attributes
- classify rules as warning, drop, or fail
- prepare reusable Lakeflow expectation definitions

### Source

`health_insurance.silver.fhir_patient`

### Production design

The final production flow will be:

Bronze  
↓  
FHIR Patient transformation  
+  
Lakeflow expectations  
↓  
Validated Silver Patient  
↓  
Gold

### Data-quality approach

FHIR contains many optional attributes. I therefore do not treat every missing
value as invalid data.

I distinguish between:

- attributes required for a usable Patient entity
- values that should satisfy logical constraints
- optional attributes whose absence should only be monitored



In [0]:
# loading the current Silver Patient dataset for quality profiling.

from pyspark.sql import functions as F

PATIENT_TABLE = "health_insurance.silver.fhir_patient"

patient_df = spark.table(PATIENT_TABLE)

print(f"Patient rows: {patient_df.count():,}")

patient_df.printSchema()

display(patient_df.limit(10))

In [0]:
# defining candidate Patient quality rules by severity.

PATIENT_WARN_RULES = {
    "medical_record_number_present":
        "medical_record_number IS NOT NULL",

    "given_name_present":
        "given_name IS NOT NULL",

    "family_name_present":
        "family_name IS NOT NULL",

    "recognized_gender":
        "gender IN ('MALE', 'FEMALE', 'OTHER', 'UNKNOWN')",

    "phone_present":
        "phone IS NOT NULL"
}


PATIENT_DROP_RULES = {
    "patient_id_present":
        "patient_id IS NOT NULL",

    "birth_date_present":
        "birth_date IS NOT NULL"
}


PATIENT_FAIL_RULES = {
    "birth_date_not_future":
        "birth_date <= current_date()",

    "age_logically_valid":
        "age BETWEEN 0 AND 120"
}

In [0]:
# measuring how many Patient rows violate each candidate quality rule.

def profile_rules(df, rules, severity):

    results = []

    total_rows = df.count()

    for rule_name, condition in rules.items():

        failed_rows = (
            df
            .filter(
                f"NOT ({condition}) OR ({condition}) IS NULL"
            )
            .count()
        )

        results.append(
            (
                rule_name,
                severity,
                condition,
                total_rows,
                failed_rows,
                round(
                    failed_rows / total_rows * 100,
                    2
                ) if total_rows else 0.0
            )
        )

    return results

In [0]:
# profiling all proposed Patient quality rules.

patient_quality_results = []

patient_quality_results += profile_rules(
    patient_df,
    PATIENT_WARN_RULES,
    "WARN"
)

patient_quality_results += profile_rules(
    patient_df,
    PATIENT_DROP_RULES,
    "DROP"
)

patient_quality_results += profile_rules(
    patient_df,
    PATIENT_FAIL_RULES,
    "FAIL"
)

In [0]:
# presenting the Patient quality profile as a structured result.

patient_quality_profile_df = spark.createDataFrame(
    patient_quality_results,
    [
        "rule_name",
        "severity",
        "constraint",
        "total_rows",
        "failed_rows",
        "failed_percentage"
    ]
)

display(
    patient_quality_profile_df
    .orderBy(
        "severity",
        F.desc("failed_percentage")
    )
)

In [0]:
# profiling the completeness of optional Patient attributes
# before deciding their final quality severity.

patient_df.select(
    F.count("*").alias("total_patients"),

    F.sum(
        F.col("medical_record_number").isNull().cast("int")
    ).alias("missing_mrn"),

    F.sum(
        F.col("given_name").isNull().cast("int")
    ).alias("missing_given_name"),

    F.sum(
        F.col("family_name").isNull().cast("int")
    ).alias("missing_family_name"),

    F.sum(
        F.col("phone").isNull().cast("int")
    ).alias("missing_phone"),

    F.sum(
        F.col("city").isNull().cast("int")
    ).alias("missing_city"),

    F.sum(
        F.col("country").isNull().cast("int")
    ).alias("missing_country")
).show()

In [0]:
# defining the finalized Patient quality contract.

PATIENT_WARN_RULES = {
    "birth_date_present":
        "birth_date IS NOT NULL",

    "medical_record_number_present":
        "medical_record_number IS NOT NULL",

    "given_name_present":
        "given_name IS NOT NULL",

    "family_name_present":
        "family_name IS NOT NULL",

    "recognized_gender":
        "gender IN ('MALE', 'FEMALE', 'OTHER', 'UNKNOWN')",

    "phone_present":
        "phone IS NOT NULL"
}


PATIENT_DROP_RULES = {
    "patient_id_present":
        "patient_id IS NOT NULL",

    "age_logically_valid":
        "age IS NULL OR age BETWEEN 0 AND 120",

    "birth_date_not_future":
        "birth_date IS NULL OR birth_date <= current_date()"
}


PATIENT_FAIL_RULES = {}

## Publish approved Patient quality rules

I have completed the Patient quality profiling and finalized the approved
WARN and DROP rules.

I now publish these rules directly into the central Unity Catalog governance
table so the Lakeflow pipeline can retrieve them dynamically.

This keeps the published Patient quality contract connected to the notebook
where I developed and validated it.

In [0]:
# converting the finalized Patient quality contract
# into rows for the central governance repository.

from pyspark.sql import functions as F

def build_rule_rows(dataset, severity, rules, description, source_notebook):

    return [
        (
            dataset,
            rule_name,
            constraint.strip(),
            severity,
            True,
            description,
            source_notebook,
            "data_engineering",
            1
        )
        for rule_name, constraint in rules.items()
    ]

In [0]:
# preparing all approved Patient rules for publication.

patient_rule_rows = []

patient_rule_rows += build_rule_rows(
    "patient",
    "WARN",
    PATIENT_WARN_RULES,
    "FHIR Patient quality monitoring rule",
    "04-data-quality/02_patient_quality_profile"
)

patient_rule_rows += build_rule_rows(
    "patient",
    "DROP",
    PATIENT_DROP_RULES,
    "FHIR Patient record validity rule",
    "04-data-quality/02_patient_quality_profile"
)

patient_rule_rows += build_rule_rows(
    "patient",
    "FAIL",
    PATIENT_FAIL_RULES,
    "Critical FHIR Patient pipeline rule",
    "04-data-quality/02_patient_quality_profile"
)

In [0]:
# creating the Patient quality-rule publication DataFrame.

patient_rules_df = (
    spark.createDataFrame(
        patient_rule_rows,
        [
            "dataset",
            "rule_name",
            "constraint",
            "severity",
            "is_active",
            "description",
            "source_notebook",
            "owner",
            "version"
        ]
    )
    .withColumn(
        "created_at",
        F.current_timestamp()
    )
    .withColumn(
        "updated_at",
        F.current_timestamp()
    )
)

display(patient_rules_df)

In [0]:
# exposing the finalized Patient rules
# as a temporary view for idempotent publishing.

patient_rules_df.createOrReplaceTempView(
    "patient_quality_rule_updates"
)

In [0]:
%sql
-- publishing the approved Patient rules
-- directly from this profiling notebook.

MERGE INTO health_insurance.governance.quality_rules AS target

USING patient_quality_rule_updates AS source

ON target.dataset = source.dataset
AND target.rule_name = source.rule_name

WHEN MATCHED THEN UPDATE SET

    target.constraint = source.constraint,
    target.severity = source.severity,
    target.is_active = source.is_active,
    target.description = source.description,
    target.source_notebook = source.source_notebook,
    target.owner = source.owner,

    target.version =
        CASE
            WHEN target.constraint <> source.constraint
              OR target.severity <> source.severity
            THEN COALESCE(target.version, 1) + 1
            ELSE target.version
        END,

    target.updated_at = source.updated_at

WHEN NOT MATCHED THEN INSERT (
    dataset,
    rule_name,
    constraint,
    severity,
    is_active,
    description,
    source_notebook,
    owner,
    version,
    created_at,
    updated_at
)

VALUES (
    source.dataset,
    source.rule_name,
    source.constraint,
    source.severity,
    source.is_active,
    source.description,
    source.source_notebook,
    source.owner,
    source.version,
    source.created_at,
    source.updated_at
);

In [0]:
%sql
-- I am verifying the Patient rules published by this notebook.

SELECT
    dataset,
    rule_name,
    severity,
    constraint,
    source_notebook,
    version,
    is_active,
    updated_at
FROM health_insurance.governance.quality_rules
WHERE dataset = 'patient'
ORDER BY severity, rule_name;